In [0]:
# Imports
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality, join_on
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, get_metrics, available_metrics, Rollup, Cube
import pyspark.sql.functions as f
from pyspark.sql.types import *
import re
import os
import sys
import time
import upc_input
import datetime as dt
import seg
from seg import profile
from pyspark.sql.window import Window
from poirot import SparkManager
from pyspark.sql import Window

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
kpi = KPI(use_sample_mart = False, apply_privacy_filters = True)

Audience Breakdown Tab
- Campaign info and uplift metrics per audience segment and offer for all 2023-2025 campaigns.
- Includes XCM, TDC, MCP SSE, SSE, and standalone EMOD, DISP, and PUSH. No standalone offsite (only a few from 2022, not included).

LOGIC:
  - Use Uplift Parquet (Sales, Units, Visits, HHPen uplift metrics), Media Metrics (Segment and Audience Subsegment Data, Engagement Metrics), Media Meas Campaign Info Tables

Top Performer Logic:
  - TDC, SSE + MCP, EMOD, Display use 800 top performer flag for product group
  - XCM uses adjusted top performer flag for product group
  - MCP 4x Regional Offer uses Adjusted Top Performer Flag


#### Audience Insights - XCM (and 2026 REM SSE)

In [0]:
single_channel_output = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/single_channel_view.csv')
single_channel_output.display()

In [0]:
# Pulling all campaign ids of XCMS and 2026 offsites (2023-2026)
single_channel_output = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/single_channel_view.csv')
xcm_campaigns = single_channel_output.filter(
  (single_channel_output['campaign_type'] == "XCM") | ((f.col("year") == 2026) & (f.col("project_name").contains("XCM")))).select("campaign_id").distinct()
xcm_camp_ids = [row['campaign_id'] for row in xcm_campaigns.orderBy(f.col("campaign_id").desc()).collect()]
print(xcm_camp_ids, len(xcm_camp_ids))

In [0]:
# Pulling all job ids of the XCM campaigs
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped').filter(f.col('modality') == 'All Modalities').filter(f.col('campaign_id').isin(xcm_camp_ids))
xcm_job_ids = [row.job_id for row in mhtv.select('job_id').distinct().collect()]
print(xcm_job_ids, len(xcm_job_ids))

In [0]:
# Pulling uplift parquet with xcm campaign and job ids
uplift_base = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/uplift/version=v2/source=azure/') \
    .filter(f.col('campaign_id').isin(xcm_camp_ids)) \
    .filter(f.col('job_id').isin(xcm_job_ids)) \
    .filter(f.col('modality') == 'All Modalities') \
    .filter(f.col("segment").like("targeting%"))
uplift_base.display()

In [0]:
uplift_base.filter(f.col("campaign_id").isin([169340, 169341])).display()

In [0]:
# Pulling uplift parquet with xcm campaign and job ids
uplift_base = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/uplift/version=v2/source=azure/') \
    .filter(f.col('campaign_id').isin(xcm_camp_ids)) \
    .filter(f.col('job_id').isin(xcm_job_ids)) \
    .filter(f.col('modality') == 'All Modalities') \
    .filter(f.col('rom') == "Kroger Only") \
    .filter(f.col("segment").like("targeting%"))

# Sales, units, visits, hhpen uplift 
uplift_xcm_sales = uplift_base.filter(f.col('metric') == 'sales')
uplift_xcm_units = uplift_base.filter(f.col('metric') == 'units')
uplift_xcm_visits = uplift_base.filter(f.col('metric') == 'visits')
uplift_xcm_hhpen = uplift_base.filter(f.col('metric') == 'hhpen')

uplift_xcm_union = uplift_xcm_sales.union(uplift_xcm_units).union(uplift_xcm_visits).union(uplift_xcm_hhpen)

display(uplift_xcm_union)


In [0]:
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

# Status is BOOKED or DELIVERED only. Filter out On Hold and Killed Campaigns
mmci = mmci.filter(f.col("status") == "Booked")
mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)

mmci = mmci.select("kpm_duplicated_id", "KPM_PROJECT_ID", "project_name", "Fiscal_Start_Year", "Fiscal_Quarter", "CAMP_START_DATE", "CAMP_END_DATE", "Status") \
    .withColumnRenamed('Fiscal_Start_Year', 'year') \
    .withColumnRenamed('Fiscal_Quarter', 'quarter') \
    .withColumnRenamed('KPM_PROJECT_ID', 'campaign_id')

mmci.display()

In [0]:
# 7/22 bugfix start here: for single channel
uplift_mmci_xcm_union = uplift_xcm_union.join(mmci, on="campaign_id", how="inner")
uplift_mmci_xcm_union = uplift_mmci_xcm_union.filter(
    (f.col("sub_segment") != "null") & (f.lower(f.col("sub_segment")) != "na")).dropDuplicates()
uplift_mmci_xcm_union.display()

In [0]:
# UPLIFT metrics w/ uplift parquet

# For XCMs, use adjusted top performer flag. Need to join mhtv table to get top performer flag for campaign and pg
uplift_mhtv_xcm = uplift_mmci_xcm_union.join(
    mhtv.select(
        "campaign_id", "job_id", "product_group", 
        "adjusted_top_performer", 
        f.col("camp_cost").alias("campaign_cost")
    ),
    on=["campaign_id", "job_id", "product_group"],
    how="inner"
)

# Filter to only adjusted top performer == top performer kro for Cross Channel
uplift_xcm_top_performer = uplift_mhtv_xcm.filter(f.col("adjusted_top_performer") == "Top-Performer_KRO")
uplift_xcm_top_performer = uplift_xcm_top_performer.withColumn(
    "segment",
    f.upper(f.substring_index("segment", "_", -1))
)
uplift_xcm_top_performer = uplift_xcm_top_performer.withColumn(
    "segment",
    f.when(f.col("segment") == "AD", "DISPLAY_AD").otherwise(f.col("segment"))
)

uplift_xcm_top_performer = uplift_xcm_top_performer.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)

uplift_xcm_top_performer = uplift_xcm_top_performer.select(
    "campaign_id", "project_name", "job_id", "product_group", "significance", "CAMP_START_DATE", "CAMP_END_DATE", "year", "quarter", "campaign_cost", "metric", "campaign_type", "segment", "sub_segment", "test_hh_count", "uplift_total", "test_total", "test_per_hh", "cont_per_hh", "uplift_per_hh", "uplift_pct", 
).dropDuplicates()

display(uplift_xcm_top_performer)

In [0]:
# Engagement Metrics w/ Media Metrics Table

media_metrics_xcm = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure/') \
    .filter(
        (f.col("campaign_type") == "XCM") &
        (f.col("modality") == "All Modalities") &
        (f.col("rom") == "Kroger Only")
    )

media_metrics_xcm = media_metrics_xcm.withColumn(
    "metrics_type",
    f.upper(f.substring_index("segment", "_", -1))
)
media_metrics_xcm = media_metrics_xcm.withColumn(
    "metrics_type",
    f.when(f.col("segment") == "AD", "DISPLAY_AD").otherwise(f.col("segment"))
)
media_metrics_xcm = media_metrics_xcm.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)

pivoted_media_metric_engagements = media_metrics_xcm.filter(f.col("metric").isNotNull()
).groupBy(
    "campaign_id", "product_group", "metrics_type", "segment", "sub_segment"
).pivot("metric").agg(f.first("value"))

display(pivoted_media_metric_engagements)

In [0]:
# Extract data from media metrics where segment is a channel (targeting_tdc, targeting_sse, ect)
segment_keywords = ["targeting", "tdc", "sse", "push", "prv", "pint", "pand", "emod", "display_ad"]
pattern = "|".join(segment_keywords).upper()

engagements_xcm_targeting = pivoted_media_metric_engagements.filter(
    f.upper(f.col("segment")).rlike(pattern)
)

engagements_xcm_targeting = engagements_xcm_targeting.select(
    "campaign_id", "product_group", "metrics_type", "segment", "sub_segment", 
    "clickthrough_rate", "open_rate", "redemption_rate", "total_redemptions_visits", "download_rate", "total_downloads"
).filter(
    f.col("segment").contains("targeting")
)

engagements_xcm_targeting = engagements_xcm_targeting.withColumn(
    "segment",
    f.upper(f.substring_index("segment", "_", -1))
)
engagements_xcm_targeting = engagements_xcm_targeting.withColumn(
    "segment",
    f.when(f.col("segment") == "AD", "DISPLAY_AD").otherwise(f.col("segment"))
)
engagements_xcm_targeting = engagements_xcm_targeting.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)
display(engagements_xcm_targeting)

In [0]:
# Join data containing uplift metrics with data containing engagement metrics via inner join
audience_breakdown_xcm = uplift_xcm_top_performer.join(
    engagements_xcm_targeting,
    on=["campaign_id", "product_group", "segment", "sub_segment"],
    how="left"
)

# Selecting only necessary columns to display
selected_cols = [
    "campaign_id", "project_name", "job_id", "product_group", "CAMP_START_DATE", "CAMP_END_DATE", "year", "quarter",
    "campaign_cost", "campaign_type", "segment", "sub_segment"
]
all_cols = audience_breakdown_xcm.columns
exclude_cols = set(["product_group", "metrics_type"])
remaining_cols = [col for col in all_cols if col not in selected_cols and col not in exclude_cols]
final_cols = selected_cols + remaining_cols

# Significance == SIgnificant
audience_breakdown_xcm = audience_breakdown_xcm.select(*final_cols).dropDuplicates()
display(audience_breakdown_xcm)

#### Audience Insights - TDC

In [0]:
kpf_closed_loop_summary_tab_melted_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab.csv')
kpf_no_closed_loop_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/no_closed_loop_tab.csv')

# Pull 2023-2025 TDC
tdc_campaign_ids_closed = kpf_closed_loop_summary_tab_melted_csv.filter(f.col("campaign_type") == "TDC").select("campaign_id").distinct()
tdc_campaign_ids_no_closed = kpf_no_closed_loop_csv.filter(f.col("campaign_type") == "Targeted Digital Coupon").select("campaign_id").distinct()

tdc_camp_ids_closed = [row['campaign_id'] for row in tdc_campaign_ids_closed.orderBy(f.col("campaign_id").desc()).collect()]
tdc_camp_ids_no_closed = [row['campaign_id'] for row in tdc_campaign_ids_no_closed.orderBy(f.col("campaign_id").desc()).collect()]

# IDs of both Closed and No Closed Loop TDC
tdc_camp_ids_combined = tdc_camp_ids_closed + tdc_camp_ids_no_closed
print(tdc_camp_ids_combined, len(tdc_camp_ids_combined))

In [0]:
# Pulling all job ids of the TDC campaigs
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped').filter(f.col('modality') == 'All Modalities').filter(f.col('campaign_id').isin(tdc_camp_ids_combined))
tdc_job_ids = [row.job_id for row in mhtv.select('job_id').distinct().collect()]
print(tdc_job_ids)

In [0]:
# Pulling uplift parquet with xcm campaign and job ids
uplift_tdc_base = spark.read.parquet(
  f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/uplift/version=v2/source=azure/') \
    .filter(f.col('campaign_id').isin(tdc_camp_ids_combined)) \
    .filter(f.col('job_id').isin(tdc_job_ids)) \
    .filter(f.col('modality') == 'All Modalities') \
    .filter(f.col("segment").isin('targeting'))

# Sales, units, visits, hhpen uplift 
uplift_tdc_sales = uplift_tdc_base.filter(f.col('metric') == 'sales')
uplift_tdc_units = uplift_tdc_base.filter(f.col('metric') == 'units')
uplift_tdc_visits = uplift_tdc_base.filter(f.col('metric') == 'visits')
uplift_tdc_hhpen = uplift_tdc_base.filter(f.col('metric') == 'hhpen')

uplift_tdc_union = uplift_tdc_sales.union(uplift_tdc_units).union(uplift_tdc_visits).union(uplift_tdc_hhpen)

display(uplift_tdc_union)

In [0]:
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

# Status is BOOKED or DELIVERED only. Filter out On Hold and Killed Campaigns
mmci = mmci.filter(f.col("status") == "Booked")
mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.select("kpm_duplicated_id", "project_name", "Fiscal_Start_Year", "Fiscal_Quarter", "CAMP_START_DATE", "CAMP_END_DATE",
    "Status") \
    .withColumnRenamed('Fiscal_Start_Year', 'year') \
    .withColumnRenamed('Fiscal_Quarter', 'quarter') \
    .withColumnRenamed('kpm_duplicated_id', 'campaign_id')

uplift_mmci_tdc_union = uplift_tdc_union.join(mmci, on="campaign_id", how="inner")
uplift_mmci_tdc_union = uplift_mmci_tdc_union.filter(
    (f.col("sub_segment") != "null") & (f.lower(f.col("sub_segment")) != "na"))
uplift_mmci_tdc_union.display()

In [0]:
# Preprocessing to pull top performers
uplift_mhtv_tdc = uplift_mmci_tdc_union.join(
    mhtv.select(
        "campaign_id", "job_id", "product_group", 
        "adjusted_top_performer", 
        f.col("camp_cost").alias("campaign_cost")
    ),
    on=["campaign_id", "job_id", "product_group"],
    how="inner"
)

uplift_mhtv_tdc = uplift_mhtv_tdc.withColumn("segment", f.lit("TDC")).select(
    "campaign_id", "project_name", "job_id", "product_group", "metric", "campaign_type", "segment", "sub_segment", "test_hh_count", 
    "uplift_total", "test_total", "test_per_hh", "cont_per_hh", "uplift_per_hh", "uplift_pct", 
    "year", "quarter", "campaign_cost", "adjusted_top_performer",  "CAMP_START_DATE", "CAMP_END_DATE", "significance"
)
display(uplift_mhtv_tdc)

In [0]:
# For TDCs, use 800's product group, or adjusted top performer flag when there is an 800's product group attached to a 4x regional offer. Need to join mhtv table to get top performer flag for campaign and pg
campaign_window = Window.partitionBy("campaign_id")
uplift_tdc_top_performer = uplift_mhtv_tdc.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception", "adjusted_top_performer").dropDuplicates()

uplift_tdc_top_performer = uplift_tdc_top_performer.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)
uplift_tdc_top_performer.display()

In [0]:
# Engagement Metrics w/ Media Metrics Table

media_metrics_tdc = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure/') \
    .filter(
        (f.col("campaign_type") == "TDC") &
        (f.col("modality") == "All Modalities") &
        (f.col("rom") == "Kroger Only")
    )

# Since we are only doing TDC, we will not have "targeting_TDC", we can assume this is static
media_metrics_tdc = media_metrics_tdc.withColumn("segment", f.lit("TDC"))
# flattening segment stream to match this column on media metrics and uplift parquet
media_metrics_tdc = media_metrics_tdc.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)

pivoted_media_metric_engagements_tdc = media_metrics_tdc.filter(f.col("metric").isNotNull()
).groupBy(
    "campaign_id", "product_group", "segment", "sub_segment"
).pivot("metric").agg(f.first("value"))

display(pivoted_media_metric_engagements_tdc)


In [0]:
# Join data containing uplift metrics with data containing engagement metrics via inner join
audience_breakdown_tdc = uplift_tdc_top_performer.join(
    pivoted_media_metric_engagements_tdc,
    on=["campaign_id", "product_group", "segment", "sub_segment"],
    how="left"
)

# Add clickthrough_rate column with NULLs
audience_breakdown_tdc = audience_breakdown_tdc.withColumn("clickthrough_rate", f.lit(None))
audience_breakdown_tdc = audience_breakdown_tdc.withColumn("open_rate", f.lit(None))
# Select exactly these cols for union
selected_cols = [
    'campaign_id',
    'project_name',
    'job_id',
    'product_group',
    'CAMP_START_DATE',
    'CAMP_END_DATE',
    'year',
    'quarter',
    'campaign_cost',
    'campaign_type',
    'segment',
    'sub_segment',
    'significance',
    'metric',
    'test_hh_count',
    'uplift_total',
    'test_total',
    'test_per_hh',
    'cont_per_hh',
    'uplift_per_hh',
    'uplift_pct',
    'clickthrough_rate',
    'open_rate',
    'redemption_rate',
    'total_redemptions_visits',
    'download_rate',
    'total_downloads'
]

# Significance == Significant
audience_breakdown_tdc = audience_breakdown_tdc.select(*selected_cols).dropDuplicates()
display(audience_breakdown_tdc)

In [0]:
# Inspecting Audience Level Campaign Data that is Missing 
from pyspark.sql import Row

# Create DataFrame from tdc_camp_ids_combined
tdc_camp_ids_df = spark.createDataFrame([Row(campaign_id=cid) for cid in tdc_camp_ids_combined])
# Find ids in tdc_camp_ids_combined not in audience_breakdown_tdc
missing_ids_df = tdc_camp_ids_df.subtract(audience_breakdown_tdc.select("campaign_id").distinct())
display(missing_ids_df)

#### Audience Insights - SSE + MCP SSE
- Only SSE + MCP SSEs. REM SSE PUSH 4xs will be handled with Cross Channel Section

In [0]:
kpf_closed_loop_summary_tab_melted_csv.filter(
  ((f.col("campaign_type") == "SSE") & (f.col("year") < 2026)) | 
  ((f.col("campaign_type") == "SSE") & (~f.col("project_name").contains("XCM SSE PUSH REM") & (f.col("year") == 2026)))
).display()

In [0]:
kpf_closed_loop_summary_tab_melted_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab.csv')
kpf_no_closed_loop_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/no_closed_loop_tab.csv')

# Pull 2023-2025 SSE
sse_campaign_ids_closed = kpf_closed_loop_summary_tab_melted_csv.filter(
  ((f.col("campaign_type") == "SSE") & (f.col("year") < 2026)) | 
  ((f.col("campaign_type") == "SSE") & (~f.col("project_name").contains("XCM SSE PUSH REM") & (f.col("year") == 2026)))).select("campaign_id").distinct()

sse_campaign_ids_no_closed = kpf_no_closed_loop_csv.filter(f.col("campaign_type") == "Single Subject Email").select("campaign_id").distinct()

sse_camp_ids_closed = [row['campaign_id'] for row in sse_campaign_ids_closed.orderBy(f.col("campaign_id").desc()).collect()]
sse_camp_ids_no_closed = [row['campaign_id'] for row in sse_campaign_ids_no_closed.orderBy(f.col("campaign_id").desc()).collect()]

# IDs of both Closed and No Closed Loop SSEs
sse_camp_ids_combined = sse_camp_ids_closed + sse_camp_ids_no_closed

print(sse_camp_ids_combined, len(sse_camp_ids_combined))

In [0]:
# Pulling all job ids of the SSE campaigs
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped').filter(f.col('modality') == 'All Modalities').filter(f.col('campaign_id').isin(sse_camp_ids_combined))
sse_job_ids = [row.job_id for row in mhtv.select('job_id').distinct().collect()]
print(sse_job_ids)

In [0]:
# Pulling uplift parquet with xcm campaign and job ids
uplift_sse_base = spark.read.parquet(
  f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/uplift/version=v2/source=azure/') \
    .filter(f.col('campaign_id').isin(sse_camp_ids_combined)) \
    .filter(f.col('job_id').isin(sse_job_ids)) \
    .filter(f.col('modality') == 'All Modalities') \
    .filter(f.col("segment").isin('targeting'))

# Sales, units, visits, hhpen uplift 
uplift_sse_sales = uplift_sse_base.filter(f.col('metric') == 'sales')
uplift_sse_units = uplift_sse_base.filter(f.col('metric') == 'units')
uplift_sse_visits = uplift_sse_base.filter(f.col('metric') == 'visits')
uplift_sse_hhpen = uplift_sse_base.filter(f.col('metric') == 'hhpen')

uplift_sse_union = uplift_sse_sales.union(uplift_sse_units).union(uplift_sse_visits).union(uplift_sse_hhpen)

uplift_sse_union.display(20)

In [0]:
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

# Status is BOOKED or DELIVERED only. Filter out On Hold and Killed Campaigns
mmci = mmci.filter(f.col("status") == "Booked")
mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.select("kpm_duplicated_id", "kpm_project_id", "project_name", "Fiscal_Start_Year", "Fiscal_Quarter", "CAMP_START_DATE", "CAMP_END_DATE",
    "Status") \
    .withColumnRenamed('Fiscal_Start_Year', 'year') \
    .withColumnRenamed('Fiscal_Quarter', 'quarter') \
    .withColumnRenamed('kpm_project_id', 'campaign_id')

uplift_mmci_sse_union = uplift_sse_union.join(mmci, on="campaign_id", how="inner")
uplift_mmci_sse_union = uplift_mmci_sse_union.filter(
    (f.col("sub_segment") != "null") & (f.lower(f.col("sub_segment")) != "na"))

uplift_mmci_sse_union.filter(f.col("year") == 2026).display()

In [0]:
# Preprocessing to pull top performers
uplift_mhtv_sse = uplift_mmci_sse_union.join(
    mhtv.select(
        "campaign_id", "job_id", "product_group", 
        "adjusted_top_performer", 
        f.col("camp_cost").alias("campaign_cost")
    ),
    on=["campaign_id", "job_id", "product_group"],
    how="inner"
)

uplift_mhtv_sse = uplift_mhtv_sse.withColumn("segment", f.lit("SSE")).select(
    "campaign_id", "project_name", "job_id", "product_group", "metric", "campaign_type", "segment", "sub_segment", "test_hh_count", 
    "uplift_total", "test_total", "test_per_hh", "cont_per_hh", "uplift_per_hh", "uplift_pct", 
    "year", "quarter", "campaign_cost", "adjusted_top_performer",  "CAMP_START_DATE", "CAMP_END_DATE", "significance"
)
display(uplift_mhtv_sse)

In [0]:
# For SSE, use 800's product group, or adjusted top performer flag when there is an 800's product group attached to a 4x regional offer. Need to join mhtv table to get top performer flag for campaign and pg
campaign_window = Window.partitionBy("campaign_id")
uplift_sse_top_performer = uplift_mhtv_sse.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception", "adjusted_top_performer").dropDuplicates()

uplift_sse_top_performer = uplift_sse_top_performer.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)
#uplift_sse_top_performer.display()

In [0]:
# Engagement Metrics w/ Media Metrics Table

media_metrics_sse = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure/') \
    .filter(
        (f.col("campaign_type") == "SSE") &
        (f.col("modality") == "All Modalities") &
        (f.col("rom") == "Kroger Only")
    )

# Since we are only doing SSE, we will not have "targeting_SSE", we can assume this is static
media_metrics_sse = media_metrics_sse.withColumn("segment", f.lit("SSE"))
# flattening segment stream to match this column on media metrics and uplift parquet
media_metrics_sse = media_metrics_sse.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)

pivoted_media_metric_engagements_sse = media_metrics_sse.filter(f.col("metric").isNotNull()
).groupBy(
    "campaign_id", "product_group", "segment", "sub_segment"
).pivot("metric").agg(f.first("value"))

#display(pivoted_media_metric_engagements_sse)

In [0]:
# Join data containing uplift metrics with data containing engagement metrics via inner join
audience_breakdown_sse = uplift_sse_top_performer.join(
    pivoted_media_metric_engagements_sse,
    on=["campaign_id", "product_group", "segment", "sub_segment"],
    how="left"
)

# Select exactly these cols for union
selected_cols = [
    'campaign_id',
    'project_name',
    'job_id',
    'product_group',
    'CAMP_START_DATE',
    'CAMP_END_DATE',
    'year',
    'quarter',
    'campaign_cost',
    'campaign_type',
    'segment',
    'sub_segment',
    'significance',
    'metric',
    'test_hh_count',
    'uplift_total',
    'test_total',
    'test_per_hh',
    'cont_per_hh',
    'uplift_per_hh',
    'uplift_pct',
    'clickthrough_rate',
    'open_rate',
    'redemption_rate',
    'total_redemptions_visits',
    'download_rate',
    'total_downloads'
]

audience_breakdown_sse = audience_breakdown_sse.select(*selected_cols).dropDuplicates()
#display(audience_breakdown_sse)

#### Audience Insights - EMOD Standalone (Until 2025)
- Change filter in cell 40 IF 2026 has a EMOD standalone 

In [0]:
kpf_closed_loop_summary_tab_melted_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab.csv')
kpf_no_closed_loop_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/no_closed_loop_tab.csv')

# Pull 2023-2025 EMOD
emod_campaign_ids_closed = kpf_closed_loop_summary_tab_melted_csv.filter(f.col("campaign_type") == "EMOD").select("campaign_id").distinct()
emod_campaign_ids_no_closed = kpf_no_closed_loop_csv.filter(f.col("campaign_type") == "Email Module").select("campaign_id").distinct()

emod_camp_ids_closed = [row['campaign_id'] for row in emod_campaign_ids_closed.orderBy(f.col("campaign_id").desc()).collect()]
emod_camp_ids_no_closed = [row['campaign_id'] for row in emod_campaign_ids_no_closed.orderBy(f.col("campaign_id").desc()).collect()]

# IDs of both Closed and No Closed Loop standalone EMOD
emod_camp_ids_combined = emod_camp_ids_closed + emod_camp_ids_no_closed

print(emod_camp_ids_combined, len(emod_camp_ids_combined))

In [0]:
# Pulling all job ids of the standalone EMOD campaigns
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped').filter(f.col('modality') == 'All Modalities').filter(f.col('campaign_id').isin(emod_camp_ids_combined))
emod_job_ids = [row.job_id for row in mhtv.select('job_id').distinct().collect()]
print(emod_job_ids)

In [0]:
# Pulling uplift parquet with emod campaign and job ids
uplift_emod_base = spark.read.parquet(
  f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/uplift/version=v2/source=azure/') \
    .filter(f.col('campaign_id').isin(emod_camp_ids_combined)) \
    .filter(f.col('job_id').isin(emod_job_ids)) \
    .filter(f.col('modality') == 'All Modalities') \
    .filter(f.col("segment").isin('targeting'))

# Sales, units, visits, hhpen uplift 
uplift_emod_sales = uplift_emod_base.filter(f.col('metric') == 'sales')
uplift_emod_units = uplift_emod_base.filter(f.col('metric') == 'units')
uplift_emod_visits = uplift_emod_base.filter(f.col('metric') == 'visits')
uplift_emod_hhpen = uplift_emod_base.filter(f.col('metric') == 'hhpen')

uplift_emod_union = uplift_emod_sales.union(uplift_emod_units).union(uplift_emod_visits).union(uplift_emod_hhpen)

In [0]:
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

# Status is BOOKED or DELIVERED only. Filter out On Hold and Killed Campaigns
mmci = mmci.filter(f.col("status") == "Booked")
mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.select("kpm_duplicated_id", "project_name", "Fiscal_Start_Year", "Fiscal_Quarter", "CAMP_START_DATE", "CAMP_END_DATE",
    "Status") \
    .withColumnRenamed('Fiscal_Start_Year', 'year') \
    .withColumnRenamed('Fiscal_Quarter', 'quarter') \
    .withColumnRenamed('kpm_duplicated_id', 'campaign_id')

uplift_mmci_emod_union = uplift_emod_union.join(mmci, on="campaign_id", how="inner")
uplift_mmci_emod_union = uplift_mmci_emod_union.filter(
    (f.col("sub_segment") != "null") & (f.lower(f.col("sub_segment")) != "na") & (f.col("year") <= 2025))
uplift_mmci_emod_union.display()

In [0]:
# Preprocessing to pull top performers
uplift_mhtv_emod = uplift_mmci_emod_union.join(
    mhtv.select(
        "campaign_id", "job_id", "product_group", 
        "adjusted_top_performer", 
        f.col("camp_cost").alias("campaign_cost")
    ),
    on=["campaign_id", "job_id", "product_group"],
    how="inner"
)

uplift_mhtv_emod = uplift_mhtv_emod.withColumn("segment", f.lit("EMOD")).select(
    "campaign_id", "project_name", "job_id", "product_group", "metric", "campaign_type", "segment", "sub_segment", "test_hh_count", 
    "uplift_total", "test_total", "test_per_hh", "cont_per_hh", "uplift_per_hh", "uplift_pct", 
    "year", "quarter", "campaign_cost", "adjusted_top_performer",  "CAMP_START_DATE", "CAMP_END_DATE", "significance"
)
#display(uplift_mhtv_emod)

In [0]:
# For EMOD, use 800's product group, or adjusted top performer flag when there is an 800's product group attached to a 4x regional offer. Need to join mhtv table to get top performer flag for campaign and pg
campaign_window = Window.partitionBy("campaign_id")
uplift_emod_top_performer = uplift_mhtv_emod.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception", "adjusted_top_performer").dropDuplicates()

uplift_emod_top_performer = uplift_emod_top_performer.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)
#uplift_emod_top_performer.display()

In [0]:
# Engagement Metrics w/ Media Metrics Table

media_metrics_emod = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure/') \
    .filter(
        (f.col("campaign_type") == "EMOD") &
        (f.col("modality") == "All Modalities") &
        (f.col("rom") == "Kroger Only")
    )

# Since we are only doing EMOD, we will not have "targeting_EMOD", we can assume this is static
media_metrics_emod = media_metrics_emod.withColumn("segment", f.lit("EMOD"))
# flattening segment stream to match this column on media metrics and uplift parquet
media_metrics_emod = media_metrics_emod.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)

pivoted_media_metric_engagements_emod = media_metrics_emod.filter(f.col("metric").isNotNull()
).groupBy(
    "campaign_id", "product_group", "segment", "sub_segment"
).pivot("metric").agg(f.first("value"))

display(pivoted_media_metric_engagements_emod)

In [0]:
# Join data containing uplift metrics with data containing engagement metrics via inner join
audience_breakdown_emod = uplift_emod_top_performer.join(
    pivoted_media_metric_engagements_emod,
    on=["campaign_id", "product_group", "segment", "sub_segment"],
    how="left"
)

# Select exactly these cols for union
selected_cols = [
    'campaign_id',
    'project_name',
    'job_id',
    'product_group',
    'CAMP_START_DATE',
    'CAMP_END_DATE',
    'year',
    'quarter',
    'campaign_cost',
    'campaign_type',
    'segment',
    'sub_segment',
    'significance',
    'metric',
    'test_hh_count',
    'uplift_total',
    'test_total',
    'test_per_hh',
    'cont_per_hh',
    'uplift_per_hh',
    'uplift_pct',
    'clickthrough_rate',
    'open_rate',
    'redemption_rate',
    'total_redemptions_visits',
    'download_rate',
    'total_downloads'
]

audience_breakdown_emod = audience_breakdown_emod.select(*selected_cols).dropDuplicates()
display(audience_breakdown_emod)

#### Audience Breakdown - DISPLAY AD and PUSH standalone
- Single Channel: DISP and PUSH standalone for closed loop, as well as no closed loop Cross-Channel DISP and PUSH channels

In [0]:
kpf_closed_loop_summary_tab_melted_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/closed_loop_summary_tab.csv')
kpf_no_closed_loop_csv = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/no_closed_loop_tab.csv')

# Pull standalone closed loop PUSH and DISP, and no closed loop PUSH and DISP (standalone + xcm)
disp_push_campaign_ids_closed = kpf_closed_loop_summary_tab_melted_csv.filter(
    ((f.col("campaign_type") == "DISPLAY_AD") | (f.col("campaign_type") == "PUSH")) & (~f.col("project_name").contains("XCM"))
).select("campaign_id").distinct()
disp_push_campaign_ids_no_closed = kpf_no_closed_loop_csv.filter(
    (f.col("campaign_type") == "Display Ad") | (f.col("campaign_type") == "Push Notifications")
).select("campaign_id").distinct()

disp_push_camp_ids_closed = [row['campaign_id'] for row in disp_push_campaign_ids_closed.orderBy(f.col("campaign_id").desc()).collect()]
disp_push_camp_ids_no_closed = [row['campaign_id'] for row in disp_push_campaign_ids_no_closed.orderBy(f.col("campaign_id").desc()).collect()]

# IDs of both Closed and No Closed Loop standalone DISP or PUSH
disp_push_camp_ids_combined = disp_push_camp_ids_closed + disp_push_camp_ids_no_closed

print(disp_push_camp_ids_combined, len(disp_push_camp_ids_combined))

In [0]:
# Pulling all job ids of the standalone DISP campaigns
mhtv = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_hist_revamped').filter(f.col('modality') == 'All Modalities').filter(f.col('campaign_id').isin(disp_push_camp_ids_combined))
disp_push_job_ids = [row.job_id for row in mhtv.select('job_id').distinct().collect()]
print(disp_push_job_ids)

In [0]:
# Pulling uplift parquet with xcm campaign and job ids
uplift_disp_push_base = spark.read.parquet(
  f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/uplift/version=v2/source=azure/') \
    .filter(f.col('campaign_id').isin(disp_push_camp_ids_combined)) \
    .filter(f.col('job_id').isin(disp_push_job_ids)) \
    .filter(f.col('modality') == 'All Modalities') \
    .filter(f.col("segment").isin('targeting'))


# Sales, units, visits, hhpen uplift 
uplift_disp_sales = uplift_disp_push_base.filter(f.col('metric') == 'sales')
uplift_disp_units = uplift_disp_push_base.filter(f.col('metric') == 'units')
uplift_disp_visits = uplift_disp_push_base.filter(f.col('metric') == 'visits')
uplift_disp_hhpen = uplift_disp_push_base.filter(f.col('metric') == 'hhpen')

uplift_disp_push_union = uplift_disp_sales.union(uplift_disp_units).union(uplift_disp_visits).union(uplift_disp_hhpen)

display(uplift_disp_push_union)

In [0]:
mmci = spark.read.parquet(f'abfss://data@sa8451midsrprd.dfs.core.windows.net/media_meas_campaign_info_v')

# Status is BOOKED or DELIVERED only. Filter out On Hold and Killed Campaigns
mmci = mmci.filter(f.col("status") == "Booked")
mmci = mmci.withColumn(
    'Fiscal_Quarter',
    f.concat(f.lit('Q'), f.substring(f.col('START_FIS_QUARTER_NAME'), 9, 1))
)
mmci = mmci.select("kpm_duplicated_id", "project_name", "Fiscal_Start_Year", "Fiscal_Quarter", "CAMP_START_DATE", "CAMP_END_DATE",
    "Status") \
    .withColumnRenamed('Fiscal_Start_Year', 'year') \
    .withColumnRenamed('Fiscal_Quarter', 'quarter') \
    .withColumnRenamed('kpm_duplicated_id', 'campaign_id')

uplift_mmci_disp_push_union = uplift_disp_push_union.join(mmci, on="campaign_id", how="inner")
uplift_mmci_disp_push_union = uplift_mmci_disp_push_union.filter(
    (f.col("sub_segment") != "null") & (f.lower(f.col("sub_segment")) != "na"))
uplift_mmci_disp_push_union.display()

In [0]:
# Preprocessing to pull top performers
uplift_mhtv_disp_push = uplift_mmci_disp_push_union.join(
    mhtv.select(
        "campaign_id", "job_id", "product_group", 
        "adjusted_top_performer", 
        f.col("camp_cost").alias("campaign_cost")
    ),
    on=["campaign_id", "job_id", "product_group"],
    how="inner"
)

uplift_mhtv_disp_push = uplift_mhtv_disp_push.withColumn("segment", f.col("campaign_type")).select(
    "campaign_id", "project_name", "job_id", "product_group", "metric", "campaign_type", "segment", "sub_segment", "test_hh_count", 
    "uplift_total", "test_total", "test_per_hh", "cont_per_hh", "uplift_per_hh", "uplift_pct", 
    "year", "quarter", "campaign_cost", "adjusted_top_performer", "CAMP_START_DATE", "CAMP_END_DATE", "significance"
)
display(uplift_mhtv_disp_push)

In [0]:
# For Non XCMs, use 800's product group, or adjusted top performer flag when there is an 800's product group attached to a 4x regional offer. Need to join mhtv table to get top performer flag for campaign and pg
campaign_window = Window.partitionBy("campaign_id")
uplift_disp_push_top_performer = uplift_mhtv_disp_push.withColumn(
    "has_800", 
    f.max(f.when(f.col("product_group").like("800%"), 1).otherwise(0)).over(campaign_window)
).withColumn(
    "is_4x_exception",
    f.max(
        f.when(f.col("product_group").like("800%") & f.col("product_group").contains("4x"), 1).otherwise(0)
    ).over(campaign_window)
).filter(
    ((f.col("is_4x_exception") == 1) & (f.col("adjusted_top_performer") == "Top-Performer_KRO")) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 1) & (f.col("product_group").like("800%"))) |
    ((f.col("is_4x_exception") == 0) & (f.col("has_800") == 0) & (f.col("adjusted_top_performer") == "Top-Performer_KRO"))
).drop("has_800", "is_4x_exception", "adjusted_top_performer").dropDuplicates()

uplift_disp_push_top_performer = uplift_disp_push_top_performer.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)
uplift_disp_push_top_performer.display()

In [0]:
media_metrics_disp_push = spark.read.parquet(f'abfss://measure@sa8451camprd.dfs.core.windows.net/reports/media_metrics/campaign/version=v2/source=azure/') \
    .filter(
        ((f.col("campaign_type") == "DISPLAY") | (f.col("campaign_type") == "PUSH")) &
        (f.col("modality") == "All Modalities") &
        (f.col("rom") == "Kroger Only")
    )

# Since we are only doing DISP or PUSH, we will not have "targeting_DISP", we can assume this is static
media_metrics_disp_push = media_metrics_disp_push.withColumn("segment", f.col("campaign_type"))
# flattening segment stream to match this column on media metrics and uplift parquet
media_metrics_disp_push = media_metrics_disp_push.withColumn(
    "sub_segment",
    f.regexp_replace(f.lower(f.col("sub_segment")), " ", "")
)

pivoted_media_metric_engagements_disp_push = media_metrics_disp_push.filter(f.col("metric").isNotNull()
).groupBy(
    "campaign_id", "product_group", "segment", "sub_segment"
).pivot("metric").agg(f.first("value"))

#display(pivoted_media_metric_engagements_disp_push)

In [0]:
# Join data containing uplift metrics with data containing engagement metrics via inner join
audience_breakdown_disp_push = uplift_disp_push_top_performer.join(
    pivoted_media_metric_engagements_disp_push,
    on=["campaign_id", "product_group", "segment", "sub_segment"],
    how="left"
)

# Add clickthrough_rate column with NULLs
audience_breakdown_disp_push = audience_breakdown_disp_push.withColumn("clickthrough_rate", f.lit(None))

# Rename push_hh_open_rate to open_rate
audience_breakdown_disp_push = audience_breakdown_disp_push.withColumnRenamed("push_hh_open_rate", "open_rate")

# Select exactly these cols for union
selected_cols = [
    'campaign_id',
    'project_name',
    'job_id',
    'product_group',
    'CAMP_START_DATE',
    'CAMP_END_DATE',
    'year',
    'quarter',
    'campaign_cost',
    'campaign_type',
    'segment',
    'sub_segment',
    'significance',
    'metric',
    'test_hh_count',
    'uplift_total',
    'test_total',
    'test_per_hh',
    'cont_per_hh',
    'uplift_per_hh',
    'uplift_pct',
    'clickthrough_rate',
    'open_rate',
    'redemption_rate',
    'total_redemptions_visits',
    'download_rate',
    'total_downloads'
]

audience_breakdown_disp_push = audience_breakdown_disp_push.select(*selected_cols).dropDuplicates()
display(audience_breakdown_disp_push)

#### Audience Breakdown Tab Final Output

In [0]:
# Final Audience Breakdown Tab
audience_breakdown_final = audience_breakdown_xcm.unionByName(audience_breakdown_tdc) \
    .unionByName(audience_breakdown_sse) \
    .unionByName(audience_breakdown_emod) \
    .unionByName(audience_breakdown_disp_push) 

display(audience_breakdown_final)

In [0]:
# Rename segment to channel for clarity, create business line column from project name, and filter to 2023 and up campaigns
audience_breakdown_renamed = audience_breakdown_final.withColumnRenamed("segment", "Channel")

# Business Line Column (Tay's Logic that I converted to Python here)
audience_breakdown_final = audience_breakdown_renamed.withColumn(
    "business_line",
    f.when(
         f.upper(f.col("project_name")).contains("LOTT"), "Lottery"
    ).when(
        f.upper(f.col("project_name")).contains("OPEN LOOP") | 
        f.upper(f.col("project_name")).contains(" OL "), "Open Loop"
    ).when(
        f.upper(f.col("project_name")).contains("LOCAL") |
        f.upper(f.col("project_name")).contains("TDC KPF") |
        f.upper(f.col("project_name")).contains("TDC SFID") |
        f.upper(f.col("project_name")).contains("TDC SFPRJ") |
        f.upper(f.col("project_name")).contains("MCP") |
        f.upper(f.col("project_name")).contains("BULK") |
        f.upper(f.col("project_name")).contains("GIFT"), "Gift"
    ).when(
        f.upper(f.col("project_name")).contains("MONEY SERVICES") |
        f.upper(f.col("project_name")).contains(" MS "), "Money Services"
    ).when(
        f.upper(f.col("project_name")).contains(" PAY "), "Kroger Pay"
    ).when(
        f.upper(f.col("project_name")).contains("KROGER WALLET"), "Kroger Wallet"
    ).when(
        f.upper(f.col("project_name")).contains(" ACH "), "Debit"
    ).when(
        f.upper(f.col("project_name")).contains(" CICO "), "Account Funding"
    ).when(
        f.upper(f.col("project_name")).contains(" SBE "), "SBE"
    ).when(
        f.upper(f.col("project_name")).contains("CREDIT"), "Credit"
    ).when(
        f.upper(f.col("project_name")).contains(" REM "), "Gift"
    ).otherwise("")
).filter(f.col("CAMP_START_DATE") >= f.lit("2023-01-01"))

audience_breakdown_final.display()

In [0]:
group_cols = [
    "campaign_id", "project_name", "job_id", "product_group", 
    "campaign_type", "Channel", "sub_segment", "metric"
]

# Earliest Camp Start Date, Latest Camp End Date per campaign
window_spec = Window.partitionBy(*group_cols)
audience_breakdown_final_grouped = audience_breakdown_final.withColumn(
    "min_camp_start_date", f.min("CAMP_START_DATE").over(window_spec)
).withColumn(
    "max_camp_end_date", f.max("CAMP_END_DATE").over(window_spec)
).filter(
    (f.col("CAMP_START_DATE") == f.col("min_camp_start_date")) &
    (f.col("CAMP_END_DATE") == f.col("max_camp_end_date"))
).drop("min_camp_start_date", "max_camp_end_date"
).withColumn(
    "sub_segment", f.initcap(f.col("sub_segment"))
)

# Set specified columns to 0 if significance is "Not significant" (NOT statistically significant)
for col_name in ["uplift_total", "test_total", "test_per_hh", "cont_per_hh", "uplift_per_hh", "uplift_pct"]:
    audience_breakdown_final_grouped = audience_breakdown_final_grouped.withColumn(
        col_name,
        f.when(f.lower(f.col("significance")) == "not significant", f.lit(0)).otherwise(f.col(col_name))
    )

display(audience_breakdown_final_grouped)

In [0]:
audience_breakdown_final_grouped.coalesce(1).write.mode("overwrite").option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/audience_breakdown_tab')

In [0]:
audience_breakdown_final_test = spark.read.option("header", "true").csv('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/audience_breakdown_tab')
display(audience_breakdown_final_test)
print(audience_breakdown_final_test.columns)

In [0]:
uplift_cols = ['uplift_total', 'test_total', 'test_per_hh', 'cont_per_hh', 'uplift_per_hh', 'uplift_pct']
group_by_cols = [c for c in audience_breakdown_final_test.columns if c not in uplift_cols and c != 'metric']
metrics = ['sales', 'units', 'visits', 'hhpen']

# Create alias names for pivoted columns: metric_stat (e.g., sales_uplift_total)
agg_exprs = []
alias_names = []
for stat in uplift_cols:
    for metric in metrics:
        agg_exprs.append(f.first(f.when(f.col("metric") == metric, f.col(stat))).alias(f"{metric}_{stat}"))
        alias_names.append(f"{metric}_{stat}")

audience_breakdown_pivoted = audience_breakdown_final_test.groupBy(group_by_cols).agg(*agg_exprs)

audience_breakdown_pivoted.display()